In [79]:
import torch
import torch.nn as nn
import model
import lancedb
import os
from hydra import initialize, initialize_config_module, initialize_config_dir, compose
from omegaconf import OmegaConf
from dotenv import load_dotenv
from datafusion import functions as f
from torch.utils.data import DataLoader, random_split
import torch.nn.functional as F
import numpy as np
from PIL import Image
from torchvision import transforms
import multiprocessing
import trackio
from tqdm.auto import tqdm
import cma
import gymnasium as gym
from torch.nn.utils import parameters_to_vector, vector_to_parameters

In [80]:
with initialize(version_base=None, config_path="conf"):
    load_dotenv()
    write_url = os.environ["TRACKIO_WRITE_URL"]
    cfg = compose(config_name="config", overrides=[f'trackio.write_url="{write_url}"'])

In [81]:
class Controller(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layer = nn.Linear(cfg.controller.state_dim+cfg.controller.hidden_dim, cfg.controller.action_dim)

    def forward(self, z, h):
        o = self.layer(torch.concat((z,h), dim=0))
        return o

In [82]:
def make_env():
    return gym.make(
        "CarRacing-v3",
        render_mode="rgb_array",
        lap_complete_percent=0.95,
        domain_randomize=False,
        continuous=True,
        max_episode_steps=cfg.dataset.max_episode_steps
    )

In [83]:
controller = Controller(cfg)
rnn = model.RNN(cfg).to(cfg.device)
rnn.load_state_dict(torch.load(f'./models/rnn-{cfg.trackio.run_name}.pt', weights_only=True))
vae = model.AutoEncoder(cfg).to(cfg.device)
vae.load_state_dict(torch.load(f'./models/vae-{cfg.trackio.run_name}.pt', weights_only=True))
vae.eval()
rnn.eval()
controller.eval()

Controller(
  (layer): Linear(in_features=288, out_features=3, bias=True)
)

In [84]:
IN_DIM = cfg.controller.state_dim + cfg.controller.hidden_dim   # 288 = z (32) + h (256)
OUT_DIM = cfg.controller.action_dim                            # 3


def preprocess(obs_np):
    # obs_np: (N, 96, 96, 3) uint8  ->  (N, 3, 64, 64) float in [0,1] on device
    obs = torch.from_numpy(obs_np).permute(0, 3, 1, 2).to(cfg.device).float()
    obs = F.interpolate(obs, size=(cfg.dataset.img_size, cfg.dataset.img_size),
                        mode="bilinear", align_corners=False)
    return obs / 255


@torch.no_grad()
def rollout_batch(solutions, envs, max_steps=None):
    # solutions: list of N flat param vectors (one per CMA candidate)
    N = len(solutions)
    if max_steps is None:
        max_steps = cfg.dataset.max_episode_steps

    # Per-candidate controller weights: layout matches parameters_to_vector -> [weight, bias]
    params = torch.tensor(np.array(solutions), dtype=torch.float32, device=cfg.device)  # (N, 867)
    W = params[:, :OUT_DIM * IN_DIM].view(N, OUT_DIM, IN_DIM)                            # (N, 3, 288)
    b = params[:, OUT_DIM * IN_DIM:]                                                     # (N, 3)

    obs, _ = envs.reset()
    obs = preprocess(obs)
    hidden = None                                                     # LSTM (h, c) for all N lanes
    h = torch.zeros(N, cfg.controller.hidden_dim, device=cfg.device)  # controller input h_t
    total = np.zeros(N, dtype=np.float64)
    active = np.ones(N, dtype=bool)                                   # lanes still in their first episode

    for _ in range(max_steps):
        z, _, _ = vae.encode(obs)                                    # (N, 32)  shared VAE, batched
        x = torch.cat([z, h], dim=-1).unsqueeze(-1)                  # (N, 288, 1)
        a = torch.bmm(W, x).squeeze(-1) + b                          # (N, 3)  per-candidate linear
        a = torch.cat([torch.tanh(a[:, :1]), torch.sigmoid(a[:, 1:])], dim=-1)  # steer[-1,1], gas/brake[0,1]

        obs_np, reward, terminated, truncated, _ = envs.step(a.cpu().numpy().astype(np.float32))
        done = terminated | truncated
        total += reward * active                                     # ignore reward once a lane has finished

        # advance the shared MDN-RNN one step; out is h_t for the next controller call
        _, _, _, hidden, out = rnn(z.unsqueeze(1), a.unsqueeze(1), hidden)  # out: (N, 1, 256)
        h = out.squeeze(1)
        obs = preprocess(obs_np)

        active = active & ~done
        if not active.any():
            break

    return total   # fitness (cumulative reward) per candidate


In [85]:
x0 = parameters_to_vector(controller.parameters()).detach().numpy()
es = cma.CMAEvolutionStrategy(x0, 0.3)

N = es.popsize                                       # one env lane per candidate
envs = gym.vector.SyncVectorEnv([make_env for _ in range(N)])

try:
    for gen in tqdm(range(cfg.controller.nb_rollouts)):
        if es.stop():
            break
        solutions = es.ask()                         # N candidate param vectors
        fitnesses = rollout_batch(solutions, envs)   # one batched episode over all N
        es.tell(solutions, [-f for f in fitnesses])  # CMA minimizes -> negate reward
        es.disp()
        print(f"gen {gen}: mean={fitnesses.mean():.1f} best={fitnesses.max():.1f}")
finally:
    envs.close()

best = es.result.xbest
vector_to_parameters(torch.tensor(best, dtype=torch.float32), controller.parameters())
torch.save(controller.state_dict(), f'./models/controller-{cfg.trackio.run_name}.pt')


(12_w,24)-aCMA-ES (mu_w=7.0,w_1=24%) in dimension 867 (seed=983504, Wed Jul  1 08:58:28 2026)


  0%|          | 0/1 [00:00<?, ?it/s]

Iterat #Fevals   function value  axis ratio  sigma  min&max std  t[m:s]
    1     24 -5.335689045936491e+00 1.0e+00 2.97e-01  3e-01  3e-01 0:34.1
gen 0: mean=-4.4 best=5.3
